In [22]:
import os, json, statistics
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from src import Questions
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind
from itertools import combinations

In [61]:
# definitions
models = [
    "gpt_4o",
    "llama31_70b_Inst",
    "llama31_8B_Inst",
    "llama32_1B_Inst",
    "qwen25_72b_Inst",
    "Qwen25_7B_Inst",
    "Qwen25_1dot5B_Inst"
]

demography_profiles = [
    "employment_sector",
    "race",
    "education",
    "sex",
    "age",
    "language",
    "nationality_group"
]
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
EXCLUDED_SCORES = ["2_13", "23_", "24_", "25_", "27_"]
EXCLUDED_WHOLE = [23, 24, 25, 27]
LLM_JUDGE_TASKS=[5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
questions = Questions().get_questions()

task_group_dict = {}
for i, k in enumerate(questions.keys()):
    qtype = questions[k]["question_type"]
    if qtype in task_group_dict:
        task_group_dict[qtype].append(i)
    else:
        task_group_dict[qtype] = [i]

stacked_model_df = []
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]
    stacked_model_df.append(score_df[task_cols])

def get_min_max_from_stacked_df(stacked_model_df, clip_llm_judge=True):
    # Stack all DataFrames into one
    combined_df = pd.concat(stacked_model_df, ignore_index=True)

    # Compute min and max for each column
    retdict = {}
    for col in combined_df.columns:
        qid = int(col.split("_")[0])
        retdict[col] = {
            'min': max(0, combined_df[col].min()) if qid in LLM_JUDGE_TASKS and clip_llm_judge else combined_df[col].min(),
            'max': min(1, combined_df[col].max()) if qid in LLM_JUDGE_TASKS and clip_llm_judge else combined_df[col].max()
        }

    return retdict

universal_scoredf_minmax = get_min_max_from_stacked_df(stacked_model_df)


In [83]:
stacked_model_df[1]

,0_0,1_0,2_14,2_13,2_12,2_11,2_10,2_9,2_8,2_7,...,27_5,27_4,27_3,27_2,27_1,27_0,5_0,7_0,10_0,28_0
0,0.584563,0.909696,10.128283,0.724865,1.356617,2.296755,4.783299,20.489018,15.062671,28.399123,...,0.752835,0.768190,0.825087,0.762324,0.780764,0.777357,NaN,NaN,NaN,NaN
1,0.584874,0.868841,8.461064,0.717026,1.279297,3.223299,4.384866,20.814393,17.518137,25.532325,...,0.714023,0.725967,0.767795,0.722531,0.725572,0.728244,0.0,1.0,0.9,0.9375
2,0.606712,0.899589,10.543812,0.547400,2.533711,2.439075,5.712265,18.444409,15.220005,25.785245,...,0.711759,0.731145,0.746429,0.727806,0.754275,0.711231,0.0,0.9,0.9,0.9300
3,0.584338,0.871538,3.144250,2.678064,0.906548,3.416567,4.315304,20.212876,14.858322,25.804511,...,0.747881,0.788095,0.806492,0.766978,0.774917,0.773769,0.0,0.9,0.9,0.9700
4,NaN,0.881350,9.941594,0.985786,1.555065,3.298781,5.392564,19.345942,15.929989,23.561390,...,0.538570,0.553662,0.457135,0.557597,0.546800,0.550527,0.0,0.9,0.9,0.9825
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,0.570476,0.828068,9.599326,1.446367,2.057204,3.429651,5.022748,19.864706,17.494429,27.090831,...,0.761697,0.822510,0.862837,0.786134,0.832638,0.808414,0.0,0.8,0.9,0.9575
95,0.608761,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.646124,0.622339,0.657959,0.614825,0.612790,0.633634,0.0,NaN,0.9,NaN
96,0.636352,0.787035,8.720605,1.190329,0.972186,3.770018,4.415082,21.965251,15.031849,27.996958,...,0.637011,0.650109,0.637550,0.640513,0.642931,0.671757,NaN,0.9,0.9,0.9275
97,0.571661,0.776099,9.151800,1.406482,2.656204,2.925483,5.007131,19.909725,18.902409,25.725958,...,0.737108,0.741174,0.772783,0.750128,0.759497,0.769262,0.8,NaN,0.9,0.9800


In [62]:
group_mm_lookup_table_definition = {
    "summarization" : [3],
    "semantics" : [0,1,4],
    "machine_translation" : [2],
    "knowledge_and_reasoning" : [5,6,7,8,9,10,11,12,13,14,15,18,19],
    "conditioned_text_gen" : [28],
    "syntaxtes_1" : [16,17],
    "syntaxtes_2" : [26],
    "qa_extraction_1" : [20],
    "qa_extraction_2" : [21,22]
}

group_mm_lookup_table = {}
for k in group_mm_lookup_table_definition:
    group_minmax = {
        "min" : None,
        "max" : None,
    }
    for qdi_full, glob_mm in universal_scoredf_minmax.items():
        q_idx = int(qdi_full.split("_")[0])
        if q_idx in group_mm_lookup_table_definition[k]:
            if group_minmax["min"] is None or glob_mm["min"] < group_minmax["min"]:
                group_minmax["min"] = glob_mm["min"]
            if group_minmax["max"] is None or group_minmax["max"] < glob_mm["max"]:
                group_minmax["max"] = glob_mm["max"]
    group_mm_lookup_table[k] = group_minmax
    
index_mm_lookup_table = {}
for k in group_mm_lookup_table_definition:
    lut = group_mm_lookup_table[k]
    for qid in group_mm_lookup_table_definition[k]:
        index_mm_lookup_table[qid] = lut

index_mm_lookup_table

{3: {'min': 0.0854747810500505, 'max': 34.92076420895521},
 0: {'min': 0.1, 'max': 0.9639792520491804},
 1: {'min': 0.1, 'max': 0.9639792520491804},
 4: {'min': 0.1, 'max': 0.9639792520491804},
 2: {'min': 0.0160045777947274, 'max': 49.13853106267194},
 5: {'min': 0, 'max': 1},
 6: {'min': 0, 'max': 1},
 7: {'min': 0, 'max': 1},
 8: {'min': 0, 'max': 1},
 9: {'min': 0, 'max': 1},
 10: {'min': 0, 'max': 1},
 11: {'min': 0, 'max': 1},
 12: {'min': 0, 'max': 1},
 13: {'min': 0, 'max': 1},
 14: {'min': 0, 'max': 1},
 15: {'min': 0, 'max': 1},
 18: {'min': 0, 'max': 1},
 19: {'min': 0, 'max': 1},
 28: {'min': 0.0, 'max': 0.995},
 16: {'min': 0, 'max': 1},
 17: {'min': 0, 'max': 1},
 26: {'min': 0.3183180592991914, 'max': 0.842668344870988},
 20: {'min': 0.1346987951807229, 'max': 0.9794661190965092},
 21: {'min': 0.11076, 'max': 0.98236},
 22: {'min': 0.11076, 'max': 0.98236}}

In [76]:
group_mm_lookup_table

{'summarization': {'min': 0.0854747810500505, 'max': 34.92076420895521},
 'semantics': {'min': 0.1, 'max': 0.9639792520491804},
 'machine_translation': {'min': 0.0160045777947274, 'max': 49.13853106267194},
 'knowledge_and_reasoning': {'min': 0, 'max': 1},
 'conditioned_text_gen': {'min': 0.0, 'max': 0.995},
 'syntaxtes_1': {'min': 0, 'max': 1},
 'syntaxtes_2': {'min': 0.3183180592991914, 'max': 0.842668344870988},
 'qa_extraction_1': {'min': 0.1346987951807229, 'max': 0.9794661190965092},
 'qa_extraction_2': {'min': 0.11076, 'max': 0.98236}}

In [51]:
scorelists_by_model = {}
for model in models:
    model_specific_scores = []
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annobj_for_df = {}
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as ao:
            annotator_object = json.load(ao)
        
        annobj_for_df["id"] = annotator_object["id"]
        annobj_for_df.update(**annotator_object["profile"])
        
        for i, (question, scores_for_question) in enumerate(annotator_object["answers"].items()):
            if not scores_for_question["answer"]:
                continue
            else:
                for qdi, scores in scores_for_question["downstream_scores"][model].items():
                    annobj_for_df[qdi] = list(scores.values())[0]
        model_specific_scores.append(annobj_for_df)
    score_df = pd.DataFrame(model_specific_scores)
    exclude_cols = ["id"] + demography_profiles
    task_cols = [col for col in score_df.columns if col not in exclude_cols]

    fullscores_per_dp = {}
    for dp in demography_profiles:
        fullscores_per_dp[dp] = {}
        for tc in task_cols:
            fullscores_per_dp[dp][tc] = {}
            if not any(tc.startswith(prefix) for prefix in EXCLUDED_SCORES):
                demo_grouped = score_df.groupby([dp])
                for demo_name, group in demo_grouped:
                    # implement here for exceptions
                    if dp == "employment_sector":
                        group = group[group["language"] == "L1"]
                    elif dp == "education":
                        group = group[group["language"] == "L1"]
                    elif dp == "sex":
                        group = group[group["language"] == "L1"]
                        
                    elif(demo_name == "Black"):
                        group = group[group["language"] == "L1"]
                    elif(demo_name == "White"):
                        group = group[group["language"] == "L1"]
                    # implement here for exceptions
                    fullscores_per_dp[dp][tc][demo_name[0]] = group[tc].dropna().to_list()
    scorelists_by_model[model] = fullscores_per_dp

In [44]:
print(scorelists_by_model['gpt_4o'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['0_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].keys())
print(scorelists_by_model['gpt_4o']['employment_sector']['1_0'].values())

dict_keys(['employment_sector', 'race', 'education', 'sex', 'age', 'language', 'nationality_group'])
dict_keys(['0_0', '1_0', '2_11', '2_3', '2_4', '2_10', '2_5', '2_2', '2_1', '2_6', '2_8', '2_14', '2_13', '2_9', '2_7', '2_0', '2_12', '3_0', '4_0', '6_0', '8_0', '9_0', '11_0', '12_0', '13_0', '14_0', '15_0', '16_0', '17_0', '18_0', '19_0', '20_0', '21_0', '22_0', '26_0', '26_1', '5_0', '7_0', '10_0', '28_0'])
dict_keys(['Engineering-STEM', 'Health-STEM', 'Humanities'])
dict_keys(['Engineering-STEM', 'Health-STEM', 'Humanities'])
dict_values([[0.9137484905985856, 0.9236874236874236, 0.8915558126084442, 0.9177601887102792, 0.9077505575075002, 0.9238525785921544, 0.9499276955924356, 0.935897435897436, 0.933806981157054, 0.9519377112738108, 0.9338833702651476, 0.911682055399438, 0.923760914227016, 0.9318429661941112, 0.9297637251714768, 0.937844611528822], [0.9076676033721396, 0.901792192278862, 0.8552496139989707, 0.9216990109982774, 0.9178814207715942, 0.939952923091704, 0.9339236156997

In [73]:
import math
from scipy.stats import f_oneway

# def get_minmax_vals(data_dict):
#     all_values = [x for group in data_dict.values() for x in group]
#     return min(all_values), max(all_values)

def compute_diversity_score(data_dict, min_val, max_val, clip_llm_judge=False):
    if len(data_dict) < 2:
        return 0  # No diversity if less than 2 elements

    list_of_values = []
    for lv in data_dict.values():
        if clip_llm_judge:
            sanity = [v for v in lv if 0 <= v <= 1]
            # sanity check because some of them are in the range 1-10. if so, scale down
            if sanity == []:
                lv = [v*0.1 for v in lv]
            lv = [v for v in lv if 0 <= v <= 1]
        if max_val == min_val:
            # All values are the same → treat as constant
            avg = 0.5  # or any neutral default
        else:
            normalized = [(x - min_val) / (max_val - min_val) for x in lv]
            avg = sum(normalized) / len(normalized)
        list_of_values.append(avg)
    
    total = []
    for i in range(len(list_of_values)):
        for j in range(i + 1, len(list_of_values)):
            total.append(abs(list_of_values[i] - list_of_values[j]))

    return sum(total) / len(total)

def compute_pairwise_t_test(data_dict, min_val, max_val, clip_llm_judge=False):
    results = {}
    for (key1, values1), (key2, values2) in combinations(data_dict.items(), 2):
        
        if clip_llm_judge:
            sanity1 = [v for v in values1 if 0 <= v <= 1]
            sanity2 = [v for v in values2 if 0 <= v <= 1]
            # sanity check because some of them are in the range 1-10. if so, scale down
            if sanity1 == [] or sanity2 == []:
                values1 = [v*0.1 for v in values1]
                values2 = [v*0.1 for v in values2]
            values1 = [v for v in values1 if 0 <= v <= 1]
            values2 = [v for v in values2 if 0 <= v <= 1]
        
        values1 = [(v - min_val) / (max_val - min_val) for v in values1]
        values2 = [(v - min_val) / (max_val - min_val) for v in values2]
        stat, p = ttest_ind(values1, values2, equal_var=False)
        results[(key1, key2)] = p
    return results

def compute_anova_score(data_dict, min_val, max_val, clip_llm_judge=False):
    # normalizer
    if max_val == min_val:
        # All values are the same across all groups
        return float("nan"), float("nan")

    groups = []
    for scores in data_dict.values():
        if clip_llm_judge:
            sanity = [v for v in scores if 0 <= v <= 1]
            if sanity == []:
                scores = [v*0.1 for v in scores]
            scores = [v for v in scores if 0 <= v <= 1]
        norm = [(x - min_val) / (max_val - min_val) for x in scores]
        groups.append(norm)
    
    # Perform one-way ANOVA
    f_statistic, p_value = f_oneway(*groups)
    return f_statistic, p_value


In [75]:
# sorry for this abomination, im too tired to think lmao

P_THRESHOLD = 0.05

divscores_model_v_taskg = {"Model" : models}
pairwise_ttest_model_v_taskg = {"Model" : models}
anova_model_v_taskg = {"Model" : models}
anova_corrected_model_v_taskg = {"Model" : models}

for tg_name, tg_idxs in task_group_dict.items():
    divscore_obj_list = []
    pairwise_ttest_obj_list = []
    anova_obj_list = []
    anova_corrected_obj_list = []
    for modl in models:
        disvscore_table_row = {"model" : modl}
        pairwise_ttest_table_row = {"model" : modl}
        anova_table_row = {"model" : modl}
        anova_corrected_table_row = {"model" : modl}

        for k in scorelists_by_model[modl].keys():
            tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
            tasks = [qdi for qdi in tasks if int(qdi.split("_")[0]) in tg_idxs]
            
            divsc_total = 0
            pairwise_t_test_significant = []
            anova_significant = []
            anova_corrected_significant = []
            
            for qdi in tasks:
                question_id = int(qdi.split("_")[0])
                clip_llm_judge = True if question_id in LLM_JUDGE_TASKS else False
                qid_min = index_mm_lookup_table[question_id]["min"]
                qid_max = index_mm_lookup_table[question_id]["max"]

                divsc_total += compute_diversity_score(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
                
                pairwise_ttest = compute_pairwise_t_test(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
                if any(v < P_THRESHOLD for v in pairwise_ttest.values()):
                    pairwise_t_test_significant.append(qdi)
                
                anova_f, anova_p = compute_anova_score(scorelists_by_model[modl][k][qdi], qid_min, qid_max, clip_llm_judge)
                if anova_p < P_THRESHOLD:
                    anova_significant.append(qdi)
                
                if anova_p * len(qdi) < P_THRESHOLD:
                    anova_corrected_significant.append(qdi)

            disvscore_table_row[k] = divsc_total/len(tasks)
            pairwise_ttest_table_row[k] = {
                "rejectable" : pairwise_t_test_significant,
                "all_tasks" : tasks,
            }
            anova_table_row[k] = {
                "rejectable" : anova_significant,
                "all_tasks" : tasks,
            }
            anova_corrected_table_row[k] = {
                "rejectable" : anova_corrected_significant,
                "all_tasks" : tasks,
            }
        
        divscore_obj_list.append(disvscore_table_row)
        pairwise_ttest_obj_list.append(pairwise_ttest_table_row)
        anova_obj_list.append(anova_table_row)
        anova_corrected_obj_list.append(anova_corrected_table_row)

    # ik bro but im paranoid
    assert(models == [m["model"] for m in divscore_obj_list])
    assert(models == [m["model"] for m in pairwise_ttest_obj_list])
    assert(models == [m["model"] for m in anova_obj_list])
    assert(models == [m["model"] for m in anova_corrected_obj_list])

    divscores_model_v_taskg[tg_name] = [statistics.mean([s for s in ls.values() if isinstance(s, float)]) for ls in divscore_obj_list]
    
    pairwise_ttest_model_v_taskg[tg_name] = []
    for ptt_by_model in pairwise_ttest_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in ptt_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        pairwise_ttest_model_v_taskg[tg_name].append(len(all_rejectable)/len(ptt_by_model["employment_sector"]["all_tasks"]))
    
    anova_model_v_taskg[tg_name] = []
    for anova_by_model in anova_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in anova_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        anova_model_v_taskg[tg_name].append(len(all_rejectable)/len(anova_by_model["employment_sector"]["all_tasks"]))
    
    anova_corrected_model_v_taskg[tg_name] = []
    for anova_corrected_by_model in anova_corrected_obj_list:
        all_rejectable = [v['rejectable'] if isinstance(v, dict) else [] for v in anova_corrected_by_model.values()]
        all_rejectable = set([item for row in all_rejectable for item in row])
        anova_corrected_model_v_taskg[tg_name].append(len(all_rejectable)/len(anova_corrected_by_model["employment_sector"]["all_tasks"]))

    def process_raw_obj(obj):
        return {k : (v if isinstance(v,str) else len(v["rejectable"])/len(v["all_tasks"])) for k, v in obj.items()}
    
    def convert_to_raw_str(input):
        if isinstance(input, str):
            return input
        elif isinstance(input, dict):
            return json.dumps(input)
    
    pairwise_ttest_obj_list_processed = [process_raw_obj(inst) for inst in pairwise_ttest_obj_list]
    anova_obj_list_processed = [process_raw_obj(inst) for inst in anova_obj_list]
    anova_corrected_obj_list_processed = [process_raw_obj(inst) for inst in anova_corrected_obj_list]
    
    pairwise_ttest_obj_list_rawstr = [{k : convert_to_raw_str(v) for k, v in dd.items()} for dd in pairwise_ttest_obj_list]
    anova_obj_list_rawstr = [{k : convert_to_raw_str(v) for k, v in dd.items()} for dd in anova_obj_list]
    anova_corrected_obj_list_rawstr = [{k : convert_to_raw_str(v) for k, v in dd.items()} for dd in anova_corrected_obj_list]

    pd.DataFrame(divscore_obj_list).to_csv(f"score_tables/divscores/divscores_{tg_name}.csv")

    pd.DataFrame(pairwise_ttest_obj_list_processed).to_csv(f"score_tables/ptt/ptt_{tg_name}.csv")
    pd.DataFrame(anova_obj_list_processed).to_csv(f"score_tables/anova/anova_{tg_name}.csv")
    pd.DataFrame(anova_corrected_obj_list_processed).to_csv(f"score_tables/anova_corrected/anova_corrected_{tg_name}.csv")

    pd.DataFrame(pairwise_ttest_obj_list_rawstr).to_csv(f"score_tables/ptt/rawstr/ptt_{tg_name}_rawstr.csv")
    pd.DataFrame(anova_obj_list_rawstr).to_csv(f"score_tables/anova/rawstr/anova_{tg_name}_rawstr.csv")
    pd.DataFrame(anova_corrected_obj_list_rawstr).to_csv(f"score_tables/anova_corrected/rawstr/anova_corrected_{tg_name}_rawstr.csv")

pd.DataFrame(divscores_model_v_taskg).to_csv(f"score_tables/divscores/divscores_model_v_taskg.csv")
pd.DataFrame(pairwise_ttest_model_v_taskg).to_csv(f"score_tables/ptt/ptt_model_v_taskg.csv")
pd.DataFrame(anova_model_v_taskg).to_csv(f"score_tables/anova/anova_model_v_taskg.csv")
pd.DataFrame(anova_corrected_model_v_taskg).to_csv(f"score_tables/anova_corrected/anova_corrected_model_v_taskg.csv")

In [7]:
# ANOVA P VALUE TESTS
REJECT_THRESHOLD = 0.05

ftest_obj_list = []
for modl in models:
    table_row = {"model" : modl}
    for k in scorelists_by_model[modl].keys():
        ftest_list=[]
        tasks = [qdi for qdi in scorelists_by_model[modl][k] if not any(qdi.startswith(prefix) for prefix in EXCLUDED_SCORES)]
        task_count = len(tasks)
        for qdi in tasks:
            f, p = compute_anova_score(scorelists_by_model[modl][k][qdi])
            ftest_list.append(f)
            adj_p = p * task_count * len(models)
            # adj_p = p * task_count
            if (adj_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task-{qdi}", k, adj_p)
        ftest_filtered = [x for x in ftest_list if not math.isnan(x)]
        table_row[k] = sum(ftest_filtered) / len(ftest_filtered) if ftest_filtered else None
    ftest_obj_list.append(table_row)

ftest_df = pd.DataFrame(ftest_obj_list)
numeric_df = ftest_df.drop(columns=['model'])
row_means = numeric_df.mean(axis=1)
ftest_df['MODEL WISE MEAN'] = row_means

column_means = numeric_df.mean()
mean_row = column_means.copy()
mean_row['model'] = 'DEMOGRAPHY MEAN'
mean_row['MODEL WISE MEAN'] = float("nan")

df = pd.concat([ftest_df, pd.DataFrame([mean_row])], ignore_index=True)
df

llama32_1B_Inst REJECTED! task-21_0 language 0.04086319039134498


,model,employment_sector,race,education,sex,age,language,nationality_group,MODEL WISE MEAN
0,gpt_4o,0.938733,1.059700,0.997535,1.255356,1.232642,0.824975,1.154868,1.066258
1,llama31_70b_Inst,1.282116,1.790050,1.187866,0.573854,1.369606,1.539203,1.662821,1.343645
2,llama31_8B_Inst,0.867621,1.341729,1.174878,1.059561,1.416391,1.531551,1.844491,1.319460
3,llama32_1B_Inst,0.910052,1.405647,0.961751,0.711733,1.118108,2.003189,1.550193,1.237239
4,qwen25_72b_Inst,1.001818,1.043760,1.086093,1.218508,1.089574,1.291942,1.171789,1.129069
5,Qwen25_7B_Inst,1.180106,1.137426,1.382765,0.965277,1.434255,1.742754,1.026724,1.267044
6,Qwen25_1dot5B_Inst,1.167202,1.304088,0.796284,1.061088,0.968358,1.497051,1.395094,1.169881
7,DEMOGRAPHY MEAN,1.049664,1.297486,1.083881,0.977911,1.232705,1.490095,1.400854,NaN


In [8]:
def format_default(val):
    if isinstance(val, float):
        return f"{round(val, 2)}"
    return str(val)

def format_to_percentage(val):
    if isinstance(val, float):
        return f"{round(val * 100, 1)}%"
    return str(val)

def df_to_latex(df, format="default"):
    formatted_df = df.copy()
    for col in formatted_df.columns:
        if format == "default":
            formatfunc = format_default
        elif(format == "percentage"):
            formatfunc = format_to_percentage
        formatted_df[col] = formatted_df[col].apply(formatfunc)
    
    results_str = formatted_df.to_latex(index=False, escape=True)
    
    # patch correction:
    results_str = results_str.replace("70b", "70B").replace("72b", "72B").replace("qwen25", "Qwen25")
    results_str = results_str.replace(r"employment\_sector", "industry").replace(r"nationality\_group", "nationality")
    
    return results_str

def df_to_latex_one_column(df, caption, format="default"):
    return r"""\begin{table*}[t]
\resizebox{\textwidth}{!}{%
""" + df_to_latex(df, format) + rf"""}}
\caption{{{caption.replace("_",r"\_")}}}
\label{{tab:{caption}}}
\end{{table*}}
"""

In [9]:
tgs = os.listdir("div_score_tables")

for tg in tgs:
    resdf = pd.read_csv(f"div_score_tables/{tg}", index_col=0)
    caption = tg.replace(".csv", "")
    print(df_to_latex_one_column(resdf, caption, format="default"))
    print("=================================")

FileNotFoundError: [Errno 2] No such file or directory: 'div_score_tables'

In [ ]:
questions

{'Sentiment refers to the emotional tone or attitude expressed in a piece of text, typically categorized as positive, negative,or neutral based on the conveyed feelings or opinions.<b>Example</b>"The address has been updated" has neutral sentiment.You are using a language model like ChatGPT to determine the sentiment of a text [TEXT] into one of possible <b>three sentiment labels: positive, negative, and neutral</b>. How would you phrase your request to the language model? <b>Your response must include the placeholder [TEXT].</b>': {'clean': 'Sentiment refers to the emotional tone or attitude expressed in a piece of text, typically categorized as positive, negative,or neutral based on the conveyed feelings or opinions.\n\nExample\n"The address has been updated" has neutral sentiment.\n\nYou are using a language model like ChatGPT to determine the sentiment of a text [TEXT] into one of possible three sentiment labels: positive, negative, and neutral.\n\nHow would you phrase your request

In [21]:
task_table = []
for i, q in enumerate(questions):
    for j in range(len(questions[q]["dataset"])):
        tc = f"{i}_{j}"
        task_obj = {
            # "qdi" : tc,
            "Group" : questions[q]["question_type"]
        }
        if not any(tc.startswith(prefix) for prefix in EXCLUDED_SCORES):
            task_name = questions[q]["nickname"]
            dataset_name = questions[q]["dataset"][j]
            if "question_" not in dataset_name:
                task_name += f" - {dataset_name.replace('.csv', '')}"
                task_name = task_name.replace("500-","").replace("100-","").replace("grammatical_classification_","")
                # print(task_name)
            
            ds_df = pd.read_csv(f"question_artefacts/test_datasets/{dataset_name}")
            
            
            task_obj["Task Name and Benchmark"] = task_name
            task_obj["Dataset Size"] = len(ds_df)
            task_obj["Eval Metric"] = questions[q]["metrics"][0]
            
            task_table.append(task_obj)

task_table = pd.DataFrame(task_table)
print(task_table.to_latex(index=False, escape=True))

\begin{tabular}{llrl}
\toprule
Group & Task Name and Benchmark & Dataset Size & Eval Metric \\
\midrule
semantics & Sentiment Analysis 3 Classes - sentiment\_3class & 500 & f1 \\
semantics & Sentiment Analysis 2 Classes - sentiment\_2class & 500 & f1 \\
machine\_translation & Machine Translation - eng\_Latn-zho\_Hans & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-rus\_Cyrl & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-spa\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-deu\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-jpn\_Jpan & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-ind\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-tgl\_Latn & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-hin\_Deva & 100 & bleu \\
machine\_translation & Machine Translation - eng\_Latn-arb\_Arab & 100 & bleu \\
machine\_translat

In [ ]:
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
question_valid_ds = []
for q in questions:
    counter = 0
    for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
        annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
        with open(annotator_filepath, "r") as  ao:
            annotator_object = json.load(ao)
        if annotator_object["answers"][q]["answer"]:
            counter += 1
    question_valid_ds.append({
        "Task" : questions[q]["nickname"],
        "Valid answers" : counter,
    })
        
question_valid_ds_df = pd.DataFrame(question_valid_ds).drop(EXCLUDED_WHOLE)
print(question_valid_ds_df.to_latex(index=False, escape=True))


\begin{tabular}{lr}
\toprule
Task & Valid answers \\
\midrule
Sentiment Analysis 3 Classes & 80 \\
Sentiment Analysis 2 Classes & 90 \\
Machine Translation & 92 \\
Dialogue Summarization & 97 \\
Topic Classification & 90 \\
Free Generation: Balonku Song & 93 \\
Free Generation: G30SPKI & 93 \\
Free Generation: Hijrah & 80 \\
Free Generation: Adhan & 84 \\
Free Generation: Karak Kaliang & 94 \\
Free Generation: Giant Drum & 90 \\
Free Generation: Scientific Letter & 83 \\
Free Generation: School Life & 97 \\
Free Generation: Olympics Medallists & 96 \\
Free Generation: Hillary Clinton & 98 \\
Free Generation: MBZUAI & 98 \\
Free Generation: Grammar Correction & 98 \\
Free Generation: Grammar Correction CoT & 98 \\
Free Generation: Arithmetic & 99 \\
Free Generation: Shakespeare-Prompt Jailbreak & 97 \\
Multiple Choice Questions & 88 \\
Extractive QA & 91 \\
MultispanQA & 92 \\
Grammar Correctness Classification & 92 \\
Linguistically-conditioned Text Generation & 79 \\
\bottomrule
\end{

In [ ]:
import pandas as pd

divsc_tab = pd.read_csv("score_tables/divscores/divscores_model_v_taskg.csv", index_col=0)
ptt_tab = pd.read_csv("score_tables/ptt/ptt_model_v_taskg.csv", index_col=0)
anova_tab = pd.read_csv("score_tables/anova/anova_model_v_taskg.csv", index_col=0)

In [ ]:
print(df_to_latex_one_column(divsc_tab, "CAPTION CAPTION CAPTION"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
Model & semantics & machine\_translation & summarization & knowledge\_and\_reasoning & syntaxes & qa\_extraction & conditioned\_text\_gen \\
\midrule
gpt\_4o & 0.06 & 0.06 & 0.04 & 0.06 & 0.05 & 0.05 & 0.09 \\
llama31\_70B\_Inst & 0.06 & 0.06 & 0.03 & 0.07 & 0.06 & 0.07 & 0.1 \\
llama31\_8B\_Inst & 0.07 & 0.06 & 0.05 & 0.07 & 0.08 & 0.06 & 0.07 \\
llama32\_1B\_Inst & 0.07 & 0.05 & 0.07 & 0.07 & 0.09 & 0.09 & 0.06 \\
Qwen25\_72B\_Inst & 0.07 & 0.06 & 0.04 & 0.08 & 0.08 & 0.04 & 0.09 \\
Qwen25\_7B\_Inst & 0.07 & 0.05 & 0.03 & 0.07 & 0.12 & 0.04 & 0.12 \\
Qwen25\_1dot5B\_Inst & 0.09 & 0.06 & 0.03 & 0.06 & 0.1 & 0.07 & 0.05 \\
\bottomrule
\end{tabular}
}
\caption{CAPTION CAPTION CAPTION}
\label{tab:CAPTION CAPTION CAPTION}
\end{table*}



In [ ]:
print(df_to_latex_one_column(ptt_tab, "PTT MAIN PTT MAIN PTT MAIN", format="percentage"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
Model & semantics & machine\_translation & summarization & knowledge\_and\_reasoning & syntaxes & qa\_extraction & conditioned\_text\_gen \\
\midrule
gpt\_4o & 66.7\% & 64.3\% & 0.0\% & 61.5\% & 100.0\% & 33.3\% & 100.0\% \\
llama31\_70B\_Inst & 33.3\% & 64.3\% & 0.0\% & 61.5\% & 75.0\% & 66.7\% & 100.0\% \\
llama31\_8B\_Inst & 33.3\% & 71.4\% & 100.0\% & 38.5\% & 100.0\% & 100.0\% & 100.0\% \\
llama32\_1B\_Inst & 33.3\% & 35.7\% & 100.0\% & 30.8\% & 100.0\% & 66.7\% & 100.0\% \\
Qwen25\_72B\_Inst & 66.7\% & 50.0\% & 0.0\% & 30.8\% & 50.0\% & 33.3\% & 0.0\% \\
Qwen25\_7B\_Inst & 100.0\% & 35.7\% & 0.0\% & 53.8\% & 50.0\% & 0.0\% & 100.0\% \\
Qwen25\_1dot5B\_Inst & 66.7\% & 64.3\% & 0.0\% & 38.5\% & 50.0\% & 100.0\% & 0.0\% \\
\bottomrule
\end{tabular}
}
\caption{PTT MAIN PTT MAIN PTT MAIN}
\label{tab:PTT MAIN PTT MAIN PTT MAIN}
\end{table*}



In [ ]:
print(df_to_latex_one_column(anova_tab, "ANOVA MAIN ANOVA MAIN ANOVA MAIN", format="percentage"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
Model & semantics & machine\_translation & summarization & knowledge\_and\_reasoning & syntaxes & qa\_extraction & conditioned\_text\_gen \\
\midrule
gpt\_4o & 33.3\% & 42.9\% & 0.0\% & 15.4\% & 0.0\% & 33.3\% & 0.0\% \\
llama31\_70B\_Inst & 33.3\% & 28.6\% & 0.0\% & 38.5\% & 50.0\% & 33.3\% & 100.0\% \\
llama31\_8B\_Inst & 33.3\% & 64.3\% & 100.0\% & 15.4\% & 75.0\% & 66.7\% & 100.0\% \\
llama32\_1B\_Inst & 0.0\% & 14.3\% & 100.0\% & 23.1\% & 75.0\% & 33.3\% & 0.0\% \\
Qwen25\_72B\_Inst & 66.7\% & 7.1\% & 0.0\% & 23.1\% & 50.0\% & 0.0\% & 0.0\% \\
Qwen25\_7B\_Inst & 100.0\% & 28.6\% & 0.0\% & 30.8\% & 50.0\% & 0.0\% & 100.0\% \\
Qwen25\_1dot5B\_Inst & 33.3\% & 35.7\% & 0.0\% & 23.1\% & 25.0\% & 66.7\% & 0.0\% \\
\bottomrule
\end{tabular}
}
\caption{ANOVA MAIN ANOVA MAIN ANOVA MAIN}
\label{tab:ANOVA MAIN ANOVA MAIN ANOVA MAIN}
\end{table*}



In [84]:
ptone_ptt = pd.read_csv("score_tables/ptone_ptt.csv", index_col=0)
pgrammar_ptt = pd.read_csv("score_tables/pgrammar_ptt.csv", index_col=0)

# Convert the numeric cols to the desired string format
num_cols = ptone_ptt.columns.difference(['model'])
combined = ptone_ptt.copy()

combined[num_cols] = (
    (ptone_ptt[num_cols] * 100).round().astype(int).astype(str) + "% / " +
    (pgrammar_ptt[num_cols] * 100).round().astype(int).astype(str) + "%"
)

In [85]:
print(df_to_latex_one_column(combined, "TONE AND GRAMMAR MAIN"))

\begin{table*}[t]
\resizebox{\textwidth}{!}{%
\begin{tabular}{llllllll}
\toprule
model & conditioned\_text\_gen & knowledge\_and\_reasoning & machine\_translation & qa\_extraction & semantics & summarization & syntaxes \\
\midrule
gpt\_4o & 0\% / 0\% & 31\% / 15\% & 86\% / 21\% & 0\% / 0\% & 33\% / 0\% & 0\% / 0\% & 0\% / 0\% \\
llama31\_70B\_Inst & 100\% / 0\% & 15\% / 8\% & 29\% / 0\% & 33\% / 0\% & 33\% / 0\% & 0\% / 0\% & 25\% / 0\% \\
llama31\_8B\_Inst & 100\% / 0\% & 8\% / 15\% & 71\% / 21\% & 0\% / 0\% & 67\% / 0\% & 0\% / 0\% & 75\% / 0\% \\
llama32\_1B\_Inst & 0\% / 0\% & 23\% / 15\% & 86\% / 57\% & 0\% / 0\% & 0\% / 0\% & 0\% / 0\% & 25\% / 0\% \\
Qwen25\_72B\_Inst & 100\% / 0\% & 38\% / 8\% & 57\% / 36\% & 0\% / 0\% & 33\% / 67\% & 0\% / 0\% & 25\% / 0\% \\
Qwen25\_7B\_Inst & 100\% / 0\% & 46\% / 8\% & 29\% / 14\% & 0\% / 33\% & 33\% / 0\% & 0\% / 0\% & 75\% / 0\% \\
Qwen25\_1dot5B\_Inst & 0\% / 0\% & 8\% / 8\% & 71\% / 14\% & 67\% / 67\% & 33\% / 0\% & 0\% / 0\% & 0\% / 50\